# Notebook 18 – ML Optimization Challenge

**Dataset:** Online Retail transactions (`data.csv`)

 ### Business problem (minimal brief)
 The warehouse wants to know **in advance which order lines will be bulk orders (12 or more units)** so it can prepare stock and offer bulk discounts. Build a model that predicts this **without using the quantity itself**.

You must independently work through 11 steps: understand the problem → metrics → baseline → models → under/overfitting → cross validation → tuning → comparison → error analysis → final model → justification.

## Step 1. Understand the Problem
- **Type:** binary classification (`1` = bulk order, quantity ≥ 12).
- **Leakage warning:** `Quantity` and `TotalPrice` (= Quantity × UnitPrice) directly reveal the answer, so they must **not** be used as features.
- **Allowed features:** `UnitPrice`, whether the customer is in the UK, and time features (`Hour`, `DayOfWeek`, `Month`).

In [11]:
import pandas as pd, numpy as np, time
from sklearn.model_selection import (train_test_split, cross_validate, cross_val_predict,
                                     RandomizedSearchCV, StratifiedKFold)
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score,
                             f1_score, confusion_matrix, precision_recall_curve)
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(8000, random_state=42)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['Month'] = df['InvoiceDate'].dt.month
df['IsUK'] = (df['Country'] == 'United Kingdom').astype(int)
X = df[['UnitPrice', 'Hour', 'DayOfWeek', 'Month', 'IsUK']]
y = (df['Quantity'] >= 12).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Bulk-order share:", round(y.mean(), 3))

Bulk-order share: 0.313


## Step 2. Select Appropriate Metrics
- About **31%** of lines are bulk orders — moderately imbalanced, so **accuracy is not enough** (always predicting "not bulk" already gives ~69%).
- **Missing a bulk order** (False Negative) loses a discount/stock-planning opportunity; a **false alarm** (False Positive) only costs a little extra preparation. So we care about **Recall**, balanced with **Precision**.
- **Chosen metrics:** **PR-AUC (Average Precision)** to compare models, **ROC-AUC** as a second view, and **Precision / Recall / F1** for the final decision.

In [12]:
print("Accuracy of 'always not-bulk':", round(1 - y_test.mean(), 3))
print("PR-AUC of a random model equals the positive share:", round(y_test.mean(), 3))

Accuracy of 'always not-bulk': 0.687
PR-AUC of a random model equals the positive share: 0.313


## Step 3. Establish a Baseline
A model that always predicts the majority class. Anything useful must beat this on PR-AUC and F1.

In [13]:
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
b_pred = baseline.predict(X_test)
print("Baseline accuracy:", round((b_pred == y_test).mean(), 3))
print("Baseline F1:      ", round(f1_score(y_test, b_pred, zero_division=0), 3))
print("Baseline PR-AUC:  ", round(average_precision_score(y_test, baseline.predict_proba(X_test)[:, 1]), 3))

Baseline accuracy: 0.687
Baseline F1:       0.0
Baseline PR-AUC:   0.313


## Step 4. Train Multiple Models
Five candidates. The unlimited Decision Tree is included on purpose to show overfitting.

In [14]:
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Decision Tree (no limit)': DecisionTreeClassifier(random_state=42),
    'Decision Tree (depth 4)': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}
for m in models.values():
    m.fit(X_train, y_train)
print("Trained:", list(models))

Trained: ['Logistic Regression', 'Decision Tree (no limit)', 'Decision Tree (depth 4)', 'Random Forest', 'Gradient Boosting']


## Step 5. Identify Underfitting / Overfitting
Compare **training PR-AUC** with **cross-validated PR-AUC** (computed on training data only — the test set stays untouched).
- Large gap → **overfitting**
- Both scores low → **possibly underfitting**

In [15]:
cv_results = {}
for name, m in models.items():
    cv_results[name] = cross_validate(m, X_train, y_train, cv=skf, scoring='average_precision',
                                      return_train_score=True)
fit_table = pd.DataFrame({
    'Model': list(models),
    'Train PR-AUC': [r['train_score'].mean() for r in cv_results.values()],
    'CV PR-AUC': [r['test_score'].mean() for r in cv_results.values()],
})
fit_table['Gap'] = fit_table['Train PR-AUC'] - fit_table['CV PR-AUC']
fit_table['Verdict'] = np.where(fit_table['Gap'] > 0.1, 'Overfitting',
                         np.where(fit_table['Train PR-AUC'] < 0.65, 'Possibly underfitting', 'Good fit'))
fit_table.round(3)

,Model,Train PR-AUC,CV PR-AUC,Gap,Verdict
0,Logistic Regression,0.634,0.633,0.001,Possibly underfitting
1,Decision Tree (no limit),0.982,0.443,0.539,Overfitting
2,Decision Tree (depth 4),0.622,0.607,0.015,Possibly underfitting
3,Random Forest,0.782,0.683,0.099,Good fit
4,Gradient Boosting,0.728,0.688,0.039,Good fit


## Step 6. Perform Cross Validation
5-fold stratified cross validation gives both the **average** score and its **spread (std)** — a smaller std means a more stable model.

In [16]:
cv_summary = pd.DataFrame({
    'Model': list(models),
    'CV PR-AUC (mean)': [r['test_score'].mean() for r in cv_results.values()],
    'CV PR-AUC (std)': [r['test_score'].std() for r in cv_results.values()],
})
cv_summary.round(3)

,Model,CV PR-AUC (mean),CV PR-AUC (std)
0,Logistic Regression,0.633,0.030
1,Decision Tree (no limit),0.443,0.011
2,Decision Tree (depth 4),0.607,0.021
3,Random Forest,0.683,0.019
4,Gradient Boosting,0.688,0.022


## Step 7. Tune Important Hyperparameters
Gradient Boosting has the best CV score so far, so we tune it with `RandomizedSearchCV` (optimizing PR-AUC).

In [17]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.7, 0.85, 1.0],
    'min_samples_leaf': [5, 20, 50],
}
search = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), param_dist, n_iter=8,
                            cv=3, scoring='average_precision', random_state=42)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)

Best params: {'subsample': 1.0, 'n_estimators': 100, 'min_samples_leaf': 5, 'max_depth': 3, 'learning_rate': 0.05}


## Step 8. Compare Models
Add the tuned model and compare everything on the **same 5 folds** using PR-AUC, plus training time. The winner is picked from this table, not from the test set.

In [18]:
all_models = dict(models)
all_models['Gradient Boosting (tuned)'] = search.best_estimator_

rows = []
for name, m in all_models.items():
    r = cross_validate(m, X_train, y_train, cv=skf, scoring=['average_precision', 'roc_auc'])
    rows.append({
        'Model': name,
        'CV PR-AUC': r['test_average_precision'].mean(),
        'CV ROC-AUC': r['test_roc_auc'].mean(),
        'Avg Fit Time (s)': r['fit_time'].mean(),
    })
comparison = pd.DataFrame(rows).sort_values('CV PR-AUC', ascending=False).round(3)
comparison

,Model,CV PR-AUC,CV ROC-AUC,Avg Fit Time (s)
4,Gradient Boosting,0.688,0.816,0.723
5,Gradient Boosting (tuned),0.684,0.812,0.767
3,Random Forest,0.683,0.812,1.488
0,Logistic Regression,0.633,0.787,0.023
2,Decision Tree (depth 4),0.607,0.790,0.012
1,Decision Tree (no limit),0.443,0.669,0.025


In [19]:
best_name = comparison.iloc[0]['Model']
final_model = all_models[best_name].fit(X_train, y_train)
print("Best model by CV PR-AUC:", best_name)

Best model by CV PR-AUC: Gradient Boosting


## Step 9. Perform Error Analysis
First choose a decision threshold using **cross-validated predictions on the training data** (maximizing F1), then apply it to the test set and study the mistakes.

In [20]:
oof = cross_val_predict(final_model, X_train, y_train, cv=skf, method='predict_proba')[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)
f1_curve = 2 * prec * rec / (prec + rec + 1e-9)
best_thr = thr[f1_curve[:-1].argmax()]

test_probs = final_model.predict_proba(X_test)[:, 1]
final_pred = (test_probs >= best_thr).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, final_pred).ravel()
print("Chosen threshold:", round(best_thr, 3))
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

Chosen threshold: 0.307
TN=803, FP=296, FN=120, TP=381


In [21]:
err = X_test.copy()
err['y'] = y_test.values
err['pred'] = final_pred
rows = []
for label, g in err.groupby('IsUK'):
    rows.append({
        'Customer': 'UK' if label == 1 else 'International',
        'Lines': len(g),
        'Recall (bulk caught)': recall_score(g['y'], g['pred']),
        'False alarm rate': ((g['pred'] == 1) & (g['y'] == 0)).sum() / (g['y'] == 0).sum(),
    })
by_group = pd.DataFrame(rows).set_index('Customer')
by_group.round(3)

,Lines,Recall (bulk caught),False alarm rate
Customer,,,
International,185,0.913,0.366
UK,1415,0.726,0.260


In [22]:
err['PriceBucket'] = pd.qcut(err['UnitPrice'], 4, duplicates='drop')
(err['y'] != err['pred']).groupby(err['PriceBucket'], observed=True).mean().round(3)

PriceBucket
(0.059, 1.25]    0.382
(1.25, 1.69]     0.290
(1.69, 3.48]     0.250
(3.48, 165.0]    0.080
dtype: float64

**What the errors show:**
- International customers' bulk orders are caught more often than UK ones, but with more false alarms.
- **Cheap items** (lowest price bucket) have the highest error rate — they are bought in both small and large quantities, so price alone can't separate them. Expensive items are much easier to classify (mostly small orders).
- **Next improvement:** add product information (e.g., `StockCode` or description keywords) since some products are typically sold in packs.

## Step 10. Select the Final Model
Only now do we use the **test set**, once, to confirm performance.

In [23]:
def report(name, probs, threshold=0.5):
    pred = (probs >= threshold).astype(int)
    return {
        'Model': name,
        'Accuracy': (pred == y_test).mean(),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'PR-AUC': average_precision_score(y_test, probs),
        'ROC-AUC': roc_auc_score(y_test, probs),
    }
pd.DataFrame([
    report('Baseline', baseline.predict_proba(X_test)[:, 1]),
    report(f'{best_name} (threshold 0.5)', test_probs),
    report(f'{best_name} (tuned threshold) – FINAL', test_probs, best_thr),
]).round(3)

,Model,Accuracy,Precision,Recall,F1,PR-AUC,ROC-AUC
0,Baseline,0.687,0.000,0.000,0.000,0.313,0.500
1,Gradient Boosting (threshold 0.5),0.783,0.755,0.455,0.568,0.707,0.822
2,Gradient Boosting (tuned threshold) – FINAL,0.740,0.563,0.760,0.647,0.707,0.822


## Step 11. Justify the Final Decision

**Final model: Gradient Boosting (default settings) with a tuned decision threshold.**

1. **Best cross-validated PR-AUC.** Gradient Boosting scored highest on the same 5 folds (0.688), narrowly ahead of Random Forest (0.683) and clearly ahead of Decision Trees and Logistic Regression. The choice was made using cross validation, **not the test set**.
2. **Generalizes well.** The unlimited Decision Tree overfits badly (train PR-AUC near 1.0 vs ~0.44 in CV). Gradient Boosting has a small train-CV gap (0.04, versus 0.10 for Random Forest), and its test score is in line with its CV score.
3. **Tuning helped little — and that's an honest finding.** The default settings were already near the best; the tuned and default versions score almost the same, so the model isn't very sensitive to these hyperparameters. The features are the real limit.
4. **Metric-driven choice, not accuracy.** At threshold 0.5 the model is precise but misses many bulk orders. Lowering the threshold raises recall from about 0.46 to 0.76 (F1 from about 0.57 to 0.65) while accuracy drops — the right trade-off because a missed bulk order costs more than a false alarm.
5. **Good balance of complexity.** It trains in a fraction of a second (faster than Random Forest) and is stronger than simpler models. A shallow Decision Tree is easier to explain, but scored clearly lower.

**Limitations:** performance is moderate (PR-AUC ≈ 0.70 vs 0.31 for random). Richer product features would help most.